In [2]:
import google.generativeai as genai
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-preview-05-2026
models/

C:\Users\Jamie Teh\AppData\Local\Temp\ipykernel_35768\523946257.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model = genai.GenerativeModel("gemini-3.5-flash")

response = model.generate_content("What is RAG in one sentence?")
print(response.text)

C:\Users\Jamie Teh\AppData\Local\Temp\ipykernel_37544\3482554157.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


**Retrieval-Augmented Generation (RAG)** is an AI framework that improves the accuracy and reliability of large language models by fetching relevant facts from an external knowledge base before generating a response.


In [6]:
chat = model.start_chat(history=[])   # Gemini keeps the history for you

while True:
    prompt = input("Enter your prompt: ")
    if prompt == "exit":
        break
    response = chat.send_message(prompt)   # automatically appends to history
    print(response.text)

Hi Jamie! Nice to meet you. How can I help you today?


In [10]:
import os
import google.generativeai as genai

BOOKS = [
    {"title": "Dune", "genre": "sci-fi", "price": 12},
    {"title": "The Hobbit", "genre": "fantasy", "price": 10},
    {"title": "Mistborn", "genre": "fantasy", "price": 14},
    {"title": "Foundation", "genre": "sci-fi", "price": 11},
]

#retrive the real data from the fake database we set up 
def search_books(title=None, genre=None, price_range=None):
    """Search the bookshop by title, genre, and/or maximum price.

    Args:
        title: partial title to match.
        genre: exact genre, e.g. 'fantasy' or 'sci-fi'.
        price_range: maximum price; returns books at or below this.
    """
    results = BOOKS
    if title:
        results = [book for book in results if title.lower() in book["title"].lower()]
    if genre:
        results = [book for book in results if genre.lower() == book["genre"].lower()]
    if price_range:
        results = [book for book in results if book["price"] <= price_range]
    return results
    

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model = genai.GenerativeModel(
    "gemini-3.5-flash",
    tools=[search_books],          # pass the REAL function, not a JSON description
)
chat = model.start_chat(enable_automatic_function_calling=True)

while True:
    prompt = input("You: ")
    if prompt == "exit":
        break
    response = chat.send_message(prompt)
    print("Assistant:", response.text)

Assistant: We have the following fantasy book under $12:

*   **The Hobbit** - $10


In [13]:
import os
import numpy as np
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

DOCS = [
    "Refunds are available within 30 days of purchase with a receipt.",
    "Standard UK shipping takes 3 to 5 business days.",
    "Orders over £50 qualify for free shipping.",
    "Our support team is available Monday to Friday, 9am to 5pm.",
]

def embed(text):
    result = genai.embed_content("models/gemini-embedding-2", content=text)
    return np.array(result["embedding"])

# PHASE 1 — index the docs once
doc_vectors = [embed(d) for d in DOCS]

model = genai.GenerativeModel("gemini-3.5-flash")

# PHASE 2 — answer a question
def ask(question):
    q_vector = embed(question)
    scores = [np.dot(q_vector, dv) for dv in doc_vectors]
    best_index = int(np.argmax(scores))
    best_doc = DOCS[best_index]
    response = model.generate_content(
        f"Answer using only this context: {best_doc}\n\nQuestion: {question}"
    )
    return response.text

print(ask("How long do I have to get my money back?"))

Based on the context provided, you have within 30 days of purchase to get your money back (with a receipt).


In [12]:
import google.generativeai as genai, os
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [15]:
####json format

import json

model = genai.GenerativeModel("models/gemini-pro-latest")
message = "Hi, I want a refund for order 4471, it arrived broken and I'm pretty annoyed."
prompt = f"""Extract the following fields from this customer message as JSON:
- order_number
- issue_type
- urgency (low, medium, or high)

Message: {message}"""

try:
    response = model.generate_content(
        prompt,
        generation_config={"response_mime_type": "application/json"},
    )
    data = json.loads(response.text)  
    print(data["order_number"])
except Exception as e:
    print("Model return with an invalid JSON")

Model return with an invalid JSON


In [8]:
import json

model = genai.GenerativeModel("models/gemini-2.5-flash")

message = "Hi, I want a refund for order 4471, it arrived broken and I'm pretty annoyed."

prompt = f"""Extract the following fields from this customer message as JSON:
- order_number
- issue_type
- urgency (low, medium, or high)

Message: {message}"""

try:
    response = model.generate_content(
        prompt,
        generation_config={"response_mime_type": "application/json"},  # the HOW
    )
    data = json.loads(response.text)        # assigned to a variable, not returned
    print(data["order_number"])
except Exception as e:
    print(f"Failed to parse: {e}")
    data = None                             # fallback so code can continue

4471


In [7]:
import os
import json

def load_documents(docs_dir="docs"):
    documents = []
    for filename in os.listdir(docs_dir):
        path = os.path.join(docs_dir, filename)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append({
            "doc_id": filename,
            "path": path,
            "text": text,
        })
    return documents

docs = load_documents()
os.makedirs("artifacts", exist_ok=True)              # create the folder if missing
with open("artifacts/documents.json", "w") as f:     # save inside artifacts/
    json.dump(docs, f, indent=2)

print(f"Loaded {len(docs)} documents")

Loaded 1 documents


In [8]:
import json

with open("artifacts/documents.json", "r") as f:
    docs = json.load(f)

CHUNK_SIZE = 200   # chunk size defined in code, as the spec requires

def chunk_documents(docs, chunk_size=CHUNK_SIZE):
    chunks = []
    for doc in docs:
        text = doc["text"]
        # walk through the text in steps of chunk_size
        for i in range(0, len(text), chunk_size):
            chunk_text = text[i : i + chunk_size]
            chunks.append({
                "chunk_id": f"{doc['doc_id']}_chunk{i // chunk_size}",  # stable id
                "doc_id": doc["doc_id"],
                "text": chunk_text,
                "start_char": i,
                "end_char": i + len(chunk_text),
            })
    return chunks

chunks = chunk_documents(docs)
with open("artifacts/chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

print(f"Created {len(chunks)} chunks")


Created 3 chunks


In [22]:
import json
import os

with open("artifacts/chunks.json", "r") as f:
    chunks = json.load(f)

EMBED_MODEL = "models/gemini-embedding-001"

def embed_chunks(chunks):
    for c in chunks:
        result = genai.embed_content(model=EMBED_MODEL, content=c["text"])
        c["embedding"] = result["embedding"]
    return chunks

chunks = embed_chunks(chunks)

with open("artifacts/embedded_chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

print(f"Embedded and saved {len(chunks)} chunks")

Embedded and saved 3 chunks


In [25]:
import json
import os
import numpy as np

with open("artifacts/chunks.json", "r") as f:
    chunks = json.load(f)
with open("docs/questions.json", "r") as f:
    questions = json.load(f)

EMBED_MODEL = "models/gemini-embedding-001"

def embed(text):
    result = genai.embed_content(model=EMBED_MODEL, content=text)
    return np.array(result["embedding"])

# embed every chunk once (indexing)
chunk_vectors = [embed(c["text"]) for c in chunks]

def retrieve(question, top_k=3):
    q_vec = embed(question)
    scores = [float(np.dot(q_vec, cv)) for cv in chunk_vectors]
    top_indices = np.argsort(scores)[::-1][:top_k]   # top 3, highest first
    return [
        {
            "chunk_id": chunks[i]["chunk_id"],
            "doc_id": chunks[i]["doc_id"],
            "score": scores[i],
            "text": chunks[i]["text"],
        }
        for i in top_indices
    ]

retrieval_results = []
for q in questions:
    retrieval_results.append({
        "id": q["id"],
        "question": q["question"],
        "retrieved_chunks": retrieve(q["question"]),
    })

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/retrieval_results.json", "w") as f:
    json.dump(retrieval_results, f, indent=2)

print(f"Saved retrieval results for {len(retrieval_results)} questions")

Saved retrieval results for 6 questions


In [6]:
import json
import os

model = genai.GenerativeModel("models/gemini-flash-latest")   # flash, not pro (quota)

with open("artifacts/retrieval_results.json", "r") as f:
    retrieval_results = json.load(f)

def generate_answer(question, retrieved_chunks):
    # build the context from the retrieved chunks, labelled with their IDs
    context = "\n".join(
        f"[{c['chunk_id']}]: {c['text']}" for c in retrieved_chunks
    )
    valid_ids = [c["chunk_id"] for c in retrieved_chunks]

    prompt = f"""Answer the question using ONLY the context below.
Rules:
- Use only information in the context. Do not make up features or policies.
- If the context does not contain the answer, set "supported" to false and give the standard not-available message.
- Cite the chunk IDs you used in "citations". Only cite IDs from this list: {valid_ids}
- If unsupported, citations must be an empty list.

Return JSON with keys: answer (string), supported (boolean), citations (list of chunk IDs).

Context:
{context}

Question: {question}"""

    try:
        response = model.generate_content(
            prompt,
            generation_config={"response_mime_type": "application/json"},
        )
        data = json.loads(response.text)
    except (json.JSONDecodeError, Exception) as e:
        print(f"Failed for question '{question}': {e}")
        data = {
            "answer": "The knowledge base does not provide enough information to answer this.",
            "supported": False,
            "citations": [],
        }
    return data

answers = []
for r in retrieval_results:
    result = generate_answer(r["question"], r["retrieved_chunks"])
    answers.append({
        "id": r["id"],
        "question": r["question"],
        "answer": result.get("answer", ""),
        "supported": result.get("supported", False),
        "citations": result.get("citations", []),
    })

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/answers.json", "w") as f:
    json.dump(answers, f, indent=2)

print(f"Generated {len(answers)} answers")

Generated 6 answers


In [5]:
# test on just the FIRST question before running all of them
test = retrieval_results[0]
result = generate_answer(test["question"], test["retrieved_chunks"])

print("Question:", test["question"])
print("Answer:", result.get("answer"))
print("Supported:", result.get("supported"))
print("Citations:", result.get("citations"))

Failed for question 'What authentication methods does the API support?': 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 38.890638671s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 38
}
]
Question: What authentication method

In [7]:
import json
import os

with open("artifacts/answers.json", "r") as f:
    answers = json.load(f)
with open("artifacts/retrieval_results.json", "r") as f:
    retrieval_results = json.load(f)

# build a lookup: question id -> set of valid chunk ids retrieved for it
valid_chunks_by_id = {
    r["id"]: {c["chunk_id"] for c in r["retrieved_chunks"]}
    for r in retrieval_results
}
# and a lookup of the retrieved chunk text, for the grounding check
chunk_text_by_id = {
    c["chunk_id"]: c["text"]
    for r in retrieval_results
    for c in r["retrieved_chunks"]
}

def validate_answer(ans):
    issues = []
    valid_ids = valid_chunks_by_id.get(ans["id"], set())

    # rule 1: every cited chunk id must exist in this question's retrieved chunks
    for cid in ans["citations"]:
        if cid not in valid_ids:
            issues.append(f"cited chunk {cid} not in retrieved chunks")

    # rule 2: if supported, must have at least one citation
    if ans["supported"] and len(ans["citations"]) == 0:
        issues.append("supported answer has no citations")

    # rule 3: if unsupported, citations must be empty
    if not ans["supported"] and len(ans["citations"]) > 0:
        issues.append("unsupported answer should have no citations")

    # rule 4: lightweight grounding check — key answer words appear in cited text
    if ans["supported"] and ans["citations"]:
        cited_text = " ".join(
            chunk_text_by_id.get(cid, "") for cid in ans["citations"]
        ).lower()
        # check a few significant words from the answer appear in the cited text
        answer_words = [w for w in ans["answer"].lower().split() if len(w) > 5]
        overlap = [w for w in answer_words if w in cited_text]
        if answer_words and len(overlap) == 0:
            issues.append("no key answer terms found in cited text")

    return {"id": ans["id"], "passed": len(issues) == 0, "issues": issues}

validation = [validate_answer(a) for a in answers]

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/citation_validation.json", "w") as f:
    json.dump(validation, f, indent=2)

failures = sum(1 for v in validation if not v["passed"])
print(f"Validated {len(validation)} answers, {failures} failures")

Validated 6 answers, 0 failures


In [8]:
import json
import os

with open("artifacts/answers.json", "r") as f:
    answers = json.load(f)
with open("artifacts/retrieval_results.json", "r") as f:
    retrieval_results = json.load(f)
with open("artifacts/citation_validation.json", "r") as f:
    validation = json.load(f)

retrieval_by_id = {r["id"]: r for r in retrieval_results}
validation_by_id = {v["id"]: v for v in validation}

final = []
for ans in answers:
    qid = ans["id"]
    retrieved = retrieval_by_id.get(qid, {})
    retrieved_chunk_ids = [c["chunk_id"] for c in retrieved.get("retrieved_chunks", [])]

    final.append({
        "id": qid,
        "question": ans["question"],
        "answer": ans["answer"],
        "supported": ans["supported"],
        "citations": ans["citations"],
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "validation": validation_by_id.get(qid, {}),
    })

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/final_answers.json", "w") as f:
    json.dump(final, f, indent=2)

print(f"Combined {len(final)} final answers")

Combined 6 final answers
